# FT09 – portable Baseline

Jeder Zug wird als Parquet-Zeile gespeichert. Vorher-/Nachher-Bilder lassen sich mit `show_frames=True` anzeigen. Level 1–5 sind die Trainingsläufe; Level 6 bleibt ein separater Test.


In [ ]:
%pip install -q "arc-agi==0.9.9" pillow matplotlib openai pyarrow

In [ ]:
from contextlib import chdir
from pathlib import Path
import os

# 1. Lokales Spiel und Darstellung
import arc_agi
import matplotlib.pyplot as plt
import numpy as np
from arc_agi import OperationMode
from arc_agi.rendering import frame_to_rgb_array
from PIL import Image

# Offline-Kopie im Projektordner (SDK 0.9.9). Zur Laufzeit wird nichts heruntergeladen.
PROJECT_DIR = Path.cwd().resolve()
ENVIRONMENTS_DIR = PROJECT_DIR / "environment_files"
RECORDINGS_DIR = PROJECT_DIR / "recordings"
ENVIRONMENTS_DIR.mkdir(parents=True, exist_ok=True)
RECORDINGS_DIR.mkdir(parents=True, exist_ok=True)

def frame_to_image(observation, scale=8):
    """Render the newest frame of an observation as a PIL image."""
    pixels = frame_to_rgb_array(
        steps=0,
        frame=observation.frame[-1],
        scale=scale,
    )
    return Image.fromarray(pixels)

def grid_to_text(frame):
    """64x64 FT09 frame -> compact text: 31 rows x 32 hex chars (1 char = 2x2 pixels) + bar."""
    board = frame[:62:2, ::2]  # rows 0-61
    bar = int((frame[63] == 12).sum())
    lines = ["".join(f"{v:x}" for v in row) for row in board]
    lines.append(f"bar: {bar}/64")
    return "\n".join(lines)

class FT09Game:
    """Convenience wrapper around the FT09 environment."""

    def __init__(self, arcade, game_id="ft09"):
        self.env = arcade.make(game_id)
        if self.env is None:
            raise RuntimeError(
                "FT09 konnte nicht geladen werden. Prüfe die Internetverbindung "
                "und die vorherigen ARC-SDK-Meldungen."
            )
        self.actions = {a.name: a for a in self.env.action_space}
        self.obs = None

    def reset(self):
        self.obs = self.env.reset()
        return self.obs

    def step(self, x, y):
        self.obs = self.env.step(
            self.actions["ACTION6"],
            data={"x": x, "y": y},
        )
        return self.obs

    def render(self, scale=8):
        return frame_to_image(self.obs, scale=scale)

    def done(self):
        return self.obs.state.name in {"WIN", "GAME_OVER"}

In [ ]:
# 2. Modell und Baseline-Prompt
from getpass import getpass
from openai import OpenAI
from openai import APITimeoutError

# Voreinstellungen für den bisherigen Modellanbieter.
API_BASE_URL = "https://chat-1.ki-awz.iisys.de/api"
MODEL = "lisa-pro"

import unicodedata
from pathlib import Path

def _clean_api_key(raw):
    key = unicodedata.normalize("NFKC", raw or "")
    key = "".join(ch for ch in key if ch not in "\u200b\u200c\u200d\ufeff\u2060")
    key = key.strip().strip('"').strip("'").strip()
    if key.lower().startswith("bearer "):
        key = key[7:].strip()
    return key

_key_file = Path.cwd() / "api_key.txt"
if _key_file.is_file():
    API_KEY = _clean_api_key(_key_file.read_text(encoding="utf-8-sig"))
    print("API-Key aus api_key.txt gelesen.")
else:
    API_KEY = _clean_api_key(getpass("API-Key:"))

if not API_KEY:
    raise ValueError(
        "Kein API-Key. Lege api_key.txt im Projektordner an "
        "und schreibe nur den Key in die erste Zeile."
    )
if set(API_KEY) <= set("\u2022\u25cf\u00b7"):
    raise ValueError(
        "api_key.txt enthält nur Passwort-Punkte, nicht den Key. "
        "Kopiere den Key von der Seite, auf der er angezeigt wird, "
        "füge ihn in api_key.txt ein, sodass du Buchstaben und Zahlen siehst, "
        "speichere und führe diese Zelle erneut aus."
    )
if not API_KEY.isascii():
    _bad = next(ch for ch in API_KEY if ord(ch) > 127)
    raise ValueError(
        f"Der API-Key enthält Sonderzeichen (erstes: "
        f"{unicodedata.name(_bad, 'UNKNOWN')}). "
        "Öffne api_key.txt im Editor, ersetze den Inhalt durch den Key "
        "aus der Quelle und speichere als reinen Text."
    )

client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
del API_KEY
print("Success")

import base64
import io


def image_to_data_uri(image):
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return "data:image/png;base64," + encoded



import re

GAME_PROMPT = """You are playing FT09. Your goal is to solve the current level.
The rules are hidden. Work them out from the board and the effects of your clicks.

Use the current screenshot, the text grid, the previous click's outcome, and
any saved memory.
When you are unsure, choose a click that tests a specific idea. Say what visible
change you expect and where, and save that prediction in MEMORY.

Each turn also includes a text grid of the same board. It has 31 lines of
32 hexadecimal characters. The character in column c of line r is the colour
index at grid position x = 2 × c, y = 2 × r (both counted from 0). The last
line, "bar: n/64", counts the cells of colour 12 in the bottom row.

On the next turn, check the predicted spot in the screenshot. The changed-cell
count tells you how much changed. Use the screenshot to check what changed and
where.  If the expected change did not happen, record the failed test in MEMORY
and try a different test. If you expected the click to solve the level and it
did not, reconsider the rule behind that prediction.

Choose exactly one click per turn. ACTION6 x y clicks a position on the 64 × 64
game grid. Use integers from 0 to 63, even when the screenshot is enlarged.
The origin is at the top left. x increases to the right and y downward.

Keep earlier findings that still matter. Label untested ideas as hypotheses.
Omit MEMORY if there is nothing useful to update.
Use no bullets, blank lines, or code fences.

Reply in plain text, with one field per line, in this exact order:
REASONING: <brief explanation of why you chose this click>
ACTION: ACTION6 x y
MEMORY: <short summary of useful findings, failed tests, and open hypotheses>
"""


def parse_answer(text):
    match = re.fullmatch(
        r"REASONING:[ \t]*([^\r\n]+)\r?\n"
        r"ACTION:[ \t]*ACTION6[ \t]+([0-9]{1,2})[ \t]+([0-9]{1,2})[ \t]*"
        r"(?:\r?\nMEMORY:[ \t]*([^\r\n]+))?",
        (text or "").strip(),
        flags=re.IGNORECASE,
    )
    if match is None:
        return None

    reasoning = match.group(1).strip()
    x, y = map(int, match.group(2, 3))
    if not reasoning or not (0 <= x < 64 and 0 <= y < 64):
        return None

    return reasoning, x, y, (match.group(4) or "").strip()


def ask_llm_for_move(image, text_grid, history, prompt, memory_note, temperature=None, format_reminder=""):
    previous_click = history[-1] if history else "none. This is the first turn."

    intro = "The attached image shows the current board.\n"
    rest = (
        f"Previous click: {previous_click}\n"
        "Choose one click."
    )
    if memory_note:
        rest += f"\nModel's memory from this run: {memory_note}"
    if format_reminder:
        rest += "\n" + format_reminder

    message = intro + f"Text grid of the current board:\n{text_grid}\n" + rest
    print(intro + "[text grid sent]\n" + rest, flush=True)
    request = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": message},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_to_data_uri(image)},
                    },
                ],
            },
        ],
    }
    if temperature is not None:
        request["temperature"] = temperature

    response = client.with_options(
        timeout=90.0,
        max_retries=1,
    ).chat.completions.create(**request)

    usage = getattr(response, "usage", None)
    completion_tokens = int(getattr(usage, "completion_tokens", 0) or 0)
    text = response.choices[0].message.content or ""
    move = parse_answer(text)
    if move is None:
        print("Invalid model reply:", repr(text), flush=True)
    return move, completion_tokens

## Data pipeline

The game loads offline from `environment_files/` (SDK 0.9.9). Each model turn becomes one Parquet row.

- Levels 1–5 are the training runs. Level 6 is held out and is not merged into the training file.
- Five attempts per level, at temperatures 0.2, 0.4, 0.6, 0.8 and 1.0.
- The step limit is `baseline_actions × 1.5`, rounded up: 65, 18, 35, 42, 98. Level 6 uses 56.
- An unparseable reply or a timeout is retried twice, then stored as an invalid step. The attempt continues.
- The memory note is cleared at the start of each attempt and sent again with the previous click on the next turn.

The system prompt is the refined FT09 prompt. `parse_answer` and `grid_to_text` are unchanged. A short check without a full level is `play_ft09_level(1, n_attempts=1, max_steps=2, show_frames=True)`.

In [ ]:
# 3. Spielschleife, Level-Auswahl und Parquet-Log
import sys
from copy import copy

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from data_pipeline import (
    ATTEMPT_TEMPERATURES,
    RENDER_SCALE,
    export_training_dataset,
    load_baseline_actions,
    run_attempt,
    step_limit,
    write_parquet,
)

DATASET_DIR = PROJECT_DIR / "dataset"
BASELINE_ACTIONS = load_baseline_actions(ENVIRONMENTS_DIR)


def build_prompt(state, history):
    """Agreed interface. The prompt text is GAME_PROMPT."""
    return GAME_PROMPT


def play_with_llm(
    game,
    level_number,
    attempt,
    temperature,
    max_steps,
    records,
    show_frames=False,
):
    print("Setze FT09 zurück …", flush=True)

    def on_step(event):
        record = event["record"]
        print(
            f"Level {level_number} · Versuch {attempt} · "
            f"Step {record['step']}/{max_steps} · T={temperature}",
            flush=True,
        )
        print("Game state:", record["state"], flush=True)
        print("Retries:", record["retries"], flush=True)
        print("Completion tokens:", record["completion_tokens"], flush=True)
        print("Previous actions:", event["history"], flush=True)
        if not record["valid"]:
            print(
                "Das Modell hat keinen gültigen Klick zurückgegeben. "
                "Der Schritt wird gespeichert und der Versuch läuft weiter.",
                flush=True,
            )
            return

        x, y = event["x"], event["y"]
        print("Reasoning:", record["reasoning"], flush=True)
        print("Model chose:", record["action"], flush=True)
        if event["new_note"]:
            print("Memory:", record["memory"], flush=True)
        print("Geänderte Zellen:", record["changed_cells"], flush=True)
        print("Level solved:", "yes" if record["level_solved"] else "no", flush=True)

        if show_frames:
            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            extent = (-0.5, 63.5, 63.5, -0.5)
            axes[0].imshow(event["image"], interpolation="nearest", extent=extent)
            axes[0].scatter(x, y, marker="x", color="lime", s=100)
            axes[0].set_title(f"Vorher · Klick ({x}, {y})")
            axes[1].imshow(game.render(scale=2), interpolation="nearest", extent=extent)
            axes[1].set_title("Nachher")
            fig.tight_layout()
            plt.show()
            plt.close(fig)

        if record["level_solved"]:
            print(f"Level gelöst nach {len(event['history']) + 1} Aktionen.", flush=True)

    def ask(image, grid_text, history, prompt, memory_note, temperature, format_reminder):
        try:
            return ask_llm_for_move(
                image,
                grid_text,
                history,
                prompt,
                memory_note,
                temperature=temperature,
                format_reminder=format_reminder,
            )
        except APITimeoutError as exc:
            raise TimeoutError("LISA timed out") from exc

    return run_attempt(
        reset=game.reset,
        render=lambda: game.render(scale=RENDER_SCALE),
        step=game.step,
        done=game.done,
        grid_text_fn=grid_to_text,
        ask=ask,
        prompt_for=build_prompt,
        level_number=level_number,
        attempt=attempt,
        temperature=temperature,
        max_steps=max_steps,
        records=records,
        on_step=on_step,
    )


os.environ["OPERATION_MODE"] = "offline"

with chdir(ENVIRONMENTS_DIR.parent):
    level_arcade = arc_agi.Arcade(
        operation_mode=OperationMode.OFFLINE,
        environments_dir=str(ENVIRONMENTS_DIR),
        recordings_dir=str(RECORDINGS_DIR),
    )


class FT09SingleLevel(FT09Game):
    def __init__(self, level_number):
        if not isinstance(level_number, int) or not 1 <= level_number <= 6:
            raise ValueError("Wähle ein Level von 1 bis 6.")
        # Lokale Metadaten können Pfade relativ zum Umgebungsordner enthalten.
        with chdir(ENVIRONMENTS_DIR.parent):
            super().__init__(level_arcade)
        self.level_number = level_number

    def reset(self):
        start = super().reset()
        # Der lokale SDK-Wrapper stellt die Spielinstanz über _game bereit.
        engine = self.env._game
        engine.set_level(self.level_number - 1)
        self.obs = copy(start)
        self.obs.frame = [engine.camera.render(engine.current_level.get_sprites())]
        return self.obs


def play_ft09_level(level_number, n_attempts=5, max_steps=None, show_frames=False):
    """Run the agreed attempts for one level and write dataset/ft09_level_<n>.parquet."""
    if not 1 <= n_attempts <= len(ATTEMPT_TEMPERATURES):
        raise ValueError(f"n_attempts must be between 1 and {len(ATTEMPT_TEMPERATURES)}.")
    limit = step_limit(level_number, BASELINE_ACTIONS) if max_steps is None else max_steps
    temperatures = ATTEMPT_TEMPERATURES[:n_attempts]
    records = []
    print(
        f"FT09 · Level {level_number} · {len(temperatures)} Versuche · "
        f"Limit {limit} · offline",
        flush=True,
    )
    if level_number == 6:
        print("Level 6 bleibt außerhalb des Trainings-Exports.", flush=True)

    level_game = FT09SingleLevel(level_number)
    for attempt, temperature in enumerate(temperatures, start=1):
        print(
            f"Versuch {attempt}/{len(temperatures)} · Temperatur {temperature}",
            flush=True,
        )
        play_with_llm(
            level_game,
            level_number,
            attempt,
            temperature,
            limit,
            records,
            show_frames=show_frames,
        )
        out = DATASET_DIR / f"ft09_level_{level_number}.parquet"
        write_parquet(records, out)
        print(f"{len(records)} Schritte gespeichert: {out}", flush=True)
    return records


In [ ]:
# Level 1 — five attempts, limit 65. Writes dataset/ft09_level_1.parquet.
level_1_records = play_ft09_level(1)

In [ ]:
# Level 2 — five attempts, limit 18. Writes dataset/ft09_level_2.parquet.
level_2_records = play_ft09_level(2)

In [ ]:
# Level 3 — five attempts, limit 35. Writes dataset/ft09_level_3.parquet.
level_3_records = play_ft09_level(3)

In [ ]:
# Level 4 — five attempts, limit 42. Writes dataset/ft09_level_4.parquet.
level_4_records = play_ft09_level(4)

In [ ]:
# Level 5 — five attempts, limit 98. Writes dataset/ft09_level_5.parquet.
level_5_records = play_ft09_level(5)

In [ ]:
# Training file for levels 1–5. Run this after those level cells. Level 6 stays out.
training_file = export_training_dataset(DATASET_DIR)
print(training_file)

In [ ]:
# Level 6 — held-out test, limit 56. Not included in the training export.
level_6_records = play_ft09_level(6)